# synent-task6-customersegmentation-rudrapatel
## Task 6: Customer Segmentation - Mall Customer Dataset
**Prepared by:** Rudra Patel (Data Science Intern)  
**Company:** Synent Technologies  
**Date:** June 2, 2026  

---

### 📌 Introduction & Problem Statement
In retail analytics, grouping customers into homogeneous cohorts based on purchasing power and spending behavior is a critical objective. It allows marketing managers to design targeted campaigns, optimize layouts, and improve customer satisfaction.

### 🎯 Objectives
1. **Explore and preprocess** the Mall Customer Dataset.
2. **Standardize numerical variables** to ensure equal feature weighting.
3. **Apply the Elbow Method** to identify the optimal number of segments (K).
4. **Train and fit K-Means Clustering**.
5. **Visualize customer cohorts** in 2D and 3D spaces.
6. **Formulate strategic marketing recommendations** based on cohort characteristics.

### 1. Environment Setup & Library Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import joblib

%matplotlib inline
sns.set_theme(style="whitegrid")

### 2. Dataset Loading & Assessment

In [ ]:
raw_path = "../data/raw/Mall_Customers.csv"
df = pd.read_csv(raw_path)
print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
print("\nMissing values per column:")
print(df.isnull().sum())
print(f"\nDuplicate rows count: {df.duplicated().sum()}")
df.head()

### 3. Feature Selection & Standard Scaling

In [ ]:
# Subsetting target columns for clustering
features = df[['Annual Income (k$)', 'Spending Score (1-100)']]

# Apply StandardScaler
scaler = StandardScaler()
scaled_array = scaler.fit_transform(features)
scaled_df = pd.DataFrame(scaled_array, columns=features.columns)
print("Scaled data preview:")
print(scaled_df.head())

### 4. Optimal Cluster Selection (Elbow Method)

In [ ]:
wcss = []
for k in range(1, 11):
    kmeans = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    kmeans.fit(scaled_df)
    wcss.append(kmeans.inertia_)

plt.figure(figsize=(10, 6))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--', color='#1E3A8A')
plt.title('The Elbow Method WCSS Curve', fontsize=14, fontweight='bold')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS (Inertia)')
plt.xticks(range(1, 11))
plt.tight_layout()
plt.savefig("../images/elbow_method.png", dpi=300)
plt.show()

### 5. K-Means Model Training & Serialization

In [ ]:
optimal_k = 5
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42, n_init=10)
kmeans.fit(scaled_df)

# Save model weights
os.makedirs("../models", exist_ok=True)
joblib.dump(kmeans, "../models/kmeans_model.joblib")

# Assign labels back
df['Cluster'] = kmeans.labels_
df.to_csv("../data/processed/segmented_customers.csv", index=False)
print("Model weights saved. Customer segment label allocations:")
print(df['Cluster'].value_counts())

### 6. Cohort Visualizations

In [ ]:
plt.figure(figsize=(12, 8))
colors = ['#1E3A8A', '#10B981', '#F59E0B', '#EF4444', '#8B5CF6']

# Plotting customer segments
sns.scatterplot(
    data=df,
    x='Annual Income (k$)',
    y='Spending Score (1-100)',
    hue='Cluster',
    palette=colors,
    s=100,
    alpha=0.8,
    edgecolor='black'
)

# Compute and plot centroids (mean of unscaled coordinates)
centroids = df.groupby('Cluster')[['Annual Income (k$)', 'Spending Score (1-100)']].mean()
plt.scatter(
    centroids['Annual Income (k$)'],
    centroids['Spending Score (1-100)'],
    s=300,
    c='black',
    marker='X',
    label='Centroids',
    edgecolor='white',
    linewidth=2
)

plt.title('Customer Segments and Centroids', fontsize=16, fontweight='bold')
plt.xlabel('Annual Income (k$)')
plt.ylabel('Spending Score (1-100)')
plt.legend(title='Customer cohorts')
plt.savefig("../images/customer_clusters_2d.png", dpi=300)
plt.show()

### 7. Segment Profile Descriptions & Business Recommendations

Based on the K-Means clustering algorithm ($K=5$), the customers have been grouped into distinct segments:

1. **Cluster 0: Standard Group**
   - *Profile:* Average annual income ($40\text{k}$ - $70\text{k}$), average spending score ($40$ - $60$).
   - *Strategy:* Keep engaged with standard marketing updates. Low-risk loyalists.
2. **Cluster 1: High Income, High Spending (Champions)**
   - *Profile:* High income ($>70\text{k}$), high spending score ($>60$).
   - *Strategy:* Target with premium loyalty rewards, early product launches, and personalized support.
3. **Cluster 2: Low Income, Low Spending**
   - *Profile:* Low income ($<40\text{k}$), low spending score ($<40$).
   - *Strategy:* Target with budget offerings, discount coupons, and price-saving catalogs.
4. **Cluster 3: Low Income, High Spending (Careless Spenders)**
   - *Profile:* Low income ($<40\text{k}$), high spending score ($>60$).
   - *Strategy:* Target with high-frequency impulse-buying deals, lifestyle messaging.
5. **Cluster 4: High Income, Low Spending (Conserving/Spared)**
   - *Profile:* High income ($>70\text{k}$), low spending score ($<40$).
   - *Strategy:* Survey to address dissatisfaction, target with catalog reminders of premium quality goods.